# LMDI lesson: how much did each factor contribute?

**Logarithmic Mean Divisia Index (LMDI-I)** is a method for answering a common analysis question:

> *Total claims went from 768M to 881M. How much of that +113M came from more customers, how much from more claims per policy, and how much from higher average claim amounts?*

LMDI splits the change of a total into **one effect per factor**. The split is **exact** (there is no leftover "residual" or "interaction" term), and it comes in two equivalent forms:

| Form | Answers | Effects combine by |
|---|---|---|
| **Additive** | "how many baht did factor *i* add?" | **sum** to $V^T - V^0$ |
| **Multiplicative** | "by what ratio did factor *i* grow the total?" | **multiply** to $V^T / V^0$ |

**Lesson plan**

1. The one formula behind everything: the logarithmic mean
2. The general LMDI formulas
3. The `LMDI` class and helpers
4. Case A: a single product of factors
5. Case B: a single sum of terms (and why LMDI is trivial there)
6. Case C: a sum of products, the real use case
7. Case D: several cells, each a sum
8. Activity, structure and intensity: the mix effect
9. How additive and multiplicative relate
10. Chained vs. one-step decomposition
11. Zero values and new products
12. Cheat sheet, pitfalls, exercises and references

Notation: period $0$ is the earlier period (the code calls it `b`, for *before*); period $T$ is the later period (`a`, for *after*).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. The logarithmic mean

For two positive numbers $a, b$:

$$L(a, b) = \frac{a - b}{\ln a - \ln b}, \qquad L(a, a) = a$$

It is a mean: it always lies between $b$ and $a$, a bit below the arithmetic mean. Its key property is a rearrangement of its own definition:

$$\boxed{L(a, b)\,\ln\frac{a}{b} = a - b}$$

LMDI uses this identity to turn a log change (which splits nicely across multiplied factors) back into an amount (which is what we want to report).

In [2]:
def log_mean(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.isclose(a, b), a, (a - b) / (np.log(a) - np.log(b)))

a, b = 880.6, 768.0   # total claims (M) in case C, section 6
print(f"L(a, b)               = {log_mean(a, b):.4f}")
print(f"arithmetic mean       = {(a + b) / 2:.4f}")
print(f"L(a, b) * ln(a / b)   = {log_mean(a, b) * np.log(a / b):.4f}")
print(f"a - b                 = {a - b:.4f}")

L(a, b)               = 823.0166
arithmetic mean       = 824.3000
L(a, b) * ln(a / b)   = 112.6000
a - b                 = 112.6000


## 2. The general LMDI formulas

The total $V$ is a **sum over cells** $k$ (products, regions, segments...), and each cell is a **product of factors** $x_{1k}, \dots, x_{nk}$:

$$V = \sum_k V_k = \sum_k x_{1k}\, x_{2k} \cdots x_{nk}$$

The effect of factor $i$ is:

$$\text{Additive:}\quad \Delta V_{x_i} = \sum_k L(V_k^T, V_k^0)\,\ln\frac{x_{ik}^T}{x_{ik}^0}$$

$$\text{Multiplicative:}\quad D_{x_i} = \exp\left(\sum_k \frac{L(V_k^T, V_k^0)}{L(V^T, V^0)}\,\ln\frac{x_{ik}^T}{x_{ik}^0}\right)$$

**Why it is exact.** Inside one cell, logs turn the product into a sum: $\sum_i \ln(x_{ik}^T / x_{ik}^0) = \ln(V_k^T / V_k^0)$. Multiply by $L(V_k^T, V_k^0)$ and the boxed identity gives $V_k^T - V_k^0$. Sum over cells and you get $V^T - V^0$. Dividing by $L(V^T, V^0)$ and taking `exp` gives the multiplicative version, which multiplies to $V^T / V^0$.

**The rule to remember.** The weight $L(V_k^T, V_k^0)$ uses the **cell value**, which must be *exactly the product of the factors you take logs of*. Getting the cell wrong is the most common LMDI bug.

## 3. The `LMDI` class and helpers

`LMDI.run` handles **one cell**. To decompose a total that has several cells, call it once per cell, then **sum** the additive effects and **multiply** the multiplicative effects over the cells.

| How y is built | `a_tot`, `b_tot` (cell value) | `y_a`, `y_b` (grand total) |
|---|---|---|
| $x_1 x_2 \cdots x_n$, one cell | `a.prod()`, `b.prod()` | omit (weight is 1) |
| $\sum_k x_{1k} \cdots x_{nk}$, several cells | `a.prod()`, `b.prod()` per cell | sum of all cell values |
| $x_1 + \dots + x_n$ | `a`, `b` (each term is its own cell) | sum of all terms |

In [3]:
class LMDI:
    """Log Mean Divisia Index (LMDI-I) decomposition for one cell and one pair of periods.

    Splits the change of a cell between an earlier period ``b`` and a later
    period ``a`` into the effect of each factor, in two forms:

    * additive: amounts that add up to the change.
    * multiplicative: growth factors that multiply to the growth.

    Call ``run`` once per cell, then sum the additive effects and multiply the
    multiplicative effects over the cells.

    Args (for every method):
        a, b: Factor values of the cell, later and earlier period (all > 0).
        a_tot, b_tot: Cell value, i.e. the product of the factors you take ln of.
            Product: ``a.prod()``. Sum: ``a`` (each term is its own cell).
        y_a, y_b: Grand total over ALL cells. Needed for the multiplicative form
            whenever there is more than one cell or y is a sum. Default: the cell.
    """

    def log_mean(self, a, b):
        """L(a, b) = (a - b) / (ln a - ln b), with the limit L(a, a) = a."""
        a, b = np.asarray(a, float), np.asarray(b, float)
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(np.isclose(a, b), a, (a - b) / (np.log(a) - np.log(b)))

    def log_change(self, a, b):
        """ln(a / b), the growth of a factor in log terms."""
        return np.log(np.asarray(a, float) / np.asarray(b, float))

    def additive(self, a, b, a_tot, b_tot):
        """L(cell) * ln(factor ratio). Summed over cells: y_a - y_b."""
        return self.log_mean(a_tot, b_tot) * self.log_change(a, b)

    def multiplicative(self, a, b, a_tot, b_tot, y_a=None, y_b=None):
        """exp(L(cell) / L(total) * ln(factor ratio)). Multiplied over cells: y_a / y_b."""
        y_a = a_tot if y_a is None else y_a
        y_b = b_tot if y_b is None else y_b
        weights = self.log_mean(a_tot, b_tot) / self.log_mean(y_a, y_b)
        return np.exp(weights * self.log_change(a, b))

    def run(self, a, b, a_tot, b_tot, y_a=None, y_b=None):
        """Return ``(additive, multiplicative)`` for one cell."""
        return (self.additive(a, b, a_tot, b_tot),
                self.multiplicative(a, b, a_tot, b_tot, y_a, y_b))

Three helpers used in the rest of the lesson:

* `compare` checks that the decomposition is exact. It also checks a property explained in section 9.
* `decompose` loops over the cells and returns per-cell tables with a `TOTAL` row.
* `summary` turns the `TOTAL` row into a report: the amount, the % share of the change, and the growth index.

In [4]:
def compare(additive, multiplicative, diff, growth):
    """Assert that additive effects sum to ``diff`` and multiplicative effects multiply to ``growth``."""
    additive, multiplicative = np.asarray(additive, float), np.asarray(multiplicative, float)
    assert np.isclose(additive.sum(), diff), f"additive failed: {additive.sum():.6g} vs {diff:.6g}"
    assert np.isclose(multiplicative.prod(), growth), f"multiplicative failed: {multiplicative.prod():.6g} vs {growth:.6g}"
    # section 9: ln(D_i) / dV_i is the same constant for every factor
    moved = ~np.isclose(additive, 0)
    assert np.allclose(np.log(multiplicative[moved]) / additive[moved], np.log(growth) / diff), "shares differ"


def decompose(d0, d1, factors, kind="product"):
    """LMDI over all cells (rows) of two periods.

    Args:
        d0, d1: DataFrames for the earlier / later period, one row per cell.
        factors: Factor columns.
        kind: "product" if each row is x1 * x2 * ..., "sum" if each row is x1 + x2 + ...

    Returns:
        ``(add, mult)`` DataFrames: one row per cell plus a TOTAL row.
    """
    if kind == "product":
        cell_b, cell_a = d0[factors].prod(axis=1), d1[factors].prod(axis=1)
    else:
        cell_b, cell_a = d0[factors], d1[factors]
    y_b, y_a = cell_b.values.sum(), cell_a.values.sum()

    lmdi, add_rows, mult_rows = LMDI(), {}, {}
    for k in d1.index:
        a, b = d1.loc[k, factors].values, d0.loc[k, factors].values
        if kind == "product":
            a_tot, b_tot = a.prod(), b.prod()
        else:
            a_tot, b_tot = a, b
        add_rows[k], mult_rows[k] = lmdi.run(a, b, a_tot, b_tot, y_a, y_b)

    add = pd.DataFrame(add_rows, index=factors).T
    mult = pd.DataFrame(mult_rows, index=factors).T
    add.loc["TOTAL"] = add.sum()
    mult.loc["TOTAL"] = mult.prod()
    compare(add.loc["TOTAL"], mult.loc["TOTAL"], y_a - y_b, y_a / y_b)
    return add, mult


def summary(add, mult):
    """Report table from the TOTAL rows of ``decompose``."""
    out = pd.DataFrame({
        "effect": add.loc["TOTAL"],
        "share_%": add.loc["TOTAL"] / add.loc["TOTAL"].sum() * 100,
        "index": mult.loc["TOTAL"],
    })
    out.loc["TOTAL"] = [out.effect.sum(), out["share_%"].sum(), out["index"].prod()]
    return out

## 4. Case A: a single product, $y = x_1 x_2 x_3 x_4$

Claim amount = customers × policies per customer × claims per policy × average claim.

There is one cell, so the weight $L(V_k)/L(V)$ equals 1.

In [5]:
df_prod = pd.DataFrame({
    "year": [2022, 2023, 2024, 2025],
    "x1":   [100_000, 106_000, 111_000, 118_000],   # customers
    "x2":   [1.30, 1.32, 1.31, 1.35],               # policies per customer
    "x3":   [0.040, 0.043, 0.041, 0.046],           # claims per policy
    "x4":   [28_000, 29_500, 31_000, 32_500],       # average claim amount
}).set_index("year")
factors = ["x1", "x2", "x3", "x4"]
df_prod["y"] = df_prod[factors].prod(axis=1)
df_prod

,x1,x2,x3,x4,y
year,,,,,
2022,100000,1.3000,0.0400,28000,"145,600,000.0000"
2023,106000,1.3200,0.0430,29500,"177,488,520.0000"
2024,111000,1.3100,0.0410,31000,"184,816,110.0000"
2025,118000,1.3500,0.0460,32500,"238,153,500.0000"


In [6]:
# One cell: treat each year as a one-row DataFrame (set_axis gives both rows the same cell label).
add, mult = decompose(df_prod.loc[[2022]], df_prod.loc[[2023]].set_axis([2022]), factors, kind="product")
summary(add, mult)

,effect,share_%,index
x1,"9,382,362.2417",29.4224,1.0600
x2,"2,458,342.8566",7.7092,1.0154
x3,"11,644,952.1081",36.5177,1.0750
x4,"8,402,862.7936",26.3507,1.0536
TOTAL,"31,888,520.0000",100.0000,1.2190


In [7]:
# With one cell, the multiplicative effect is just each factor's own growth ratio.
print("multiplicative:", mult.loc["TOTAL"].values.round(6))
print("x_T / x_0     :", (df_prod.loc[2023, factors] / df_prod.loc[2022, factors]).values.round(6))

multiplicative: [1.06     1.015385 1.075    1.053571]
x_T / x_0     : [1.06     1.015385 1.075    1.053571]


**How to read it**

* `effect`: the amount (in units of y) that each factor added. The amounts add up exactly to $\Delta y$.
* `share_%`: the factor's share of the change. Shares can be negative or above 100% when factors push in opposite directions.
* `index`: a growth factor. An `index` of 1.06 means "this factor alone would have grown y by 6%". The indexes multiply to $y^T / y^0$.

With a single cell, the multiplicative form gives nothing new (it is just $x^T / x^0$). The additive form is the useful part: it converts those ratios into amounts that add up.

## 5. Case B: a single sum, $y = x_1 + x_2 + x_3 + x_4$

Total claims = Term Life + Whole Life + Health + Other (all in millions).

A sum is a sum of cells, where each cell has a single factor. So each **term is its own cell**: pass `a, b` as the cell values, and the grand totals as `y_a, y_b`.

In [8]:
df_sum = pd.DataFrame({
    "year": [2022, 2023, 2024, 2025],
    "x1":   [432.0, 490.0, 502.6, 614.3],   # Term Life claims
    "x2":   [165.0, 165.2, 196.3, 201.5],   # Whole Life claims
    "x3":   [336.0, 390.6, 474.0, 551.8],   # Health claims
    "x4":   [ 40.0,  42.0,  41.0,  47.0],   # Other / riders claims
}).set_index("year")
df_sum["y"] = df_sum[factors].sum(axis=1)

add, mult = decompose(df_sum.loc[[2022]], df_sum.loc[[2023]].set_axis([2022]), factors, kind="sum")
summary(add, mult)

,effect,share_%,index
x1,58.0000,50.5226,1.0580
x2,0.2000,0.1742,1.0002
x3,54.6000,47.5610,1.0545
x4,2.0000,1.7422,1.0019
TOTAL,114.8000,100.0000,1.1180


In [9]:
print("additive effects:", add.loc["TOTAL"].values.round(6))
print("raw differences :", (df_sum.loc[2023, factors] - df_sum.loc[2022, factors]).values.round(6))

additive effects: [58.   0.2 54.6  2. ]
raw differences : [58.   0.2 54.6  2. ]


**Lesson:** for a pure sum, the additive LMDI effect is just the raw difference $x^T - x^0$, since $L(a,b)\ln(a/b) = a-b$. **You do not need LMDI to decompose a sum.** The multiplicative form still gives a consistent growth index per term, which you can use if you need to report ratios.

**Pitfall:** do *not* pass `a.sum()` as the cell value for a sum. The cell must equal the product of the factors you take logs of, and $x_1 + x_2 + \dots$ is not that product.

## 6. Case C: a sum of products, the real use case

$$\text{Claims} = \sum_{k \in \text{products}} \underbrace{x_{1k}}_{\text{customers}}\cdot \underbrace{x_{2k}}_{\text{policies/cust}}\cdot \underbrace{x_{3k}}_{\text{claims/policy}}\cdot \underbrace{x_{4k}}_{\text{avg claim}}$$

Now each product (Term Life, Health) is a cell. We want **one effect per factor across all products**. This is the case LMDI was built for.

**Why not decompose each product separately and add?** For the additive form, that is exactly what LMDI does: per-cell effects simply add. For the multiplicative form it is not: the weight $L(V_k)/L(V)$ needs the **grand total**, so the cells cannot be handled independently.

In [10]:
rows = [
    # year, product,    x1,     x2,   x3,     x4
    (2022, "Term Life", 60_000, 1.20, 0.0120, 500_000),
    (2022, "Health",    80_000, 1.50, 0.3500,   8_000),
    (2023, "Term Life", 63_000, 1.22, 0.0125, 510_000),
    (2023, "Health",    86_000, 1.52, 0.3600,   8_300),
]
df_cells = pd.DataFrame(rows, columns=["year", "product", *factors]).set_index(["year", "product"])
df_cells["y"] = df_cells[factors].prod(axis=1)
df_cells

x1     x2     x3      x4                y
year product                                                
2022 Term Life  60000 1.2000 0.0120  500000 432,000,000.0000
     Health     80000 1.5000 0.3500    8000 336,000,000.0000
2023 Term Life  63000 1.2200 0.0125  510000 489,982,500.0000
     Health     86000 1.5200 0.3600    8300 390,591,360.0000

In [11]:
add_c, mult_c = decompose(df_cells.loc[2022], df_cells.loc[2023], factors, kind="product")
print("Additive effects per product (sum over rows = TOTAL):")
display(add_c)
print("Multiplicative effects per product (product over rows = TOTAL):")
display(mult_c)

Additive effects per product (sum over rows = TOTAL):


,x1,x2,x3,x4
Term Life,"22,462,155.5509","7,609,807.3842","18,793,746.7811","9,116,790.2838"
Health,"26,224,270.3901","4,802,865.4617","10,215,071.0262","13,349,153.1219"
TOTAL,"48,686,425.9410","12,412,672.8460","29,008,817.8074","22,465,943.4057"


Multiplicative effects per product (product over rows = TOTAL):


,x1,x2,x3,x4
Term Life,1.0277,1.0093,1.0231,1.0111
Health,1.0324,1.0059,1.0125,1.0164
TOTAL,1.0609,1.0152,1.0359,1.0277


In [12]:
summary(add_c, mult_c)

,effect,share_%,index
x1,"48,686,425.9410",43.2484,1.0609
x2,"12,412,672.8460",11.0262,1.0152
x3,"29,008,817.8074",25.7687,1.0359
x4,"22,465,943.4057",19.9566,1.0277
TOTAL,"112,573,860.0000",100.0000,1.1466


**How to read it**

* The **per-product additive rows have meaning**: "within Term Life, average claim amount added this much". Each row adds up to that product's own change in claims.
* The **per-product multiplicative rows have no meaning on their own**. They are weighted by the product's share of the *grand* total. Only the `TOTAL` row is interpretable.

Note that the factors here are averaged across very different products (a Term Life average claim is around 500k, a Health claim around 8k). LMDI weights each product's factor change by the size of that product's claims. If the *mix* of customers moves between products, x1 hides it. That is the topic of section 8.

## 7. Case D: several cells, each a sum

Each product's claims = death + hospital + critical illness + other. As in case B, each term is its own cell, so the additive effects are just the raw differences, summed over products.

In [13]:
rows = [
    # year, product,    x1 (death), x2 (hospital), x3 (critical), x4 (other)  -- million baht
    (2022, "Term Life", 380.0,  30.0, 14.0,  8.0),
    (2022, "Health",     40.0, 210.0, 78.0,  8.0),
    (2023, "Term Life", 430.0,  34.0, 16.0, 10.0),
    (2023, "Health",     44.0, 245.0, 90.0, 11.6),
]
df_terms = pd.DataFrame(rows, columns=["year", "product", *factors]).set_index(["year", "product"])

add_d, mult_d = decompose(df_terms.loc[2022], df_terms.loc[2023], factors, kind="sum")
summary(add_d, mult_d)

,effect,share_%,index
x1,54.0000,47.9574,1.0678
x2,39.0000,34.6359,1.0485
x3,14.0000,12.4334,1.0172
x4,5.6000,4.9734,1.0068
TOTAL,112.6000,100.0000,1.1466


## 8. Activity, structure and intensity: the mix effect

Classic index decomposition analysis (IDA) rewrites the customers of each product as **total customers × the product's share**:

$$\text{Claims} = \sum_k \underbrace{Q}_{\text{activity}} \cdot \underbrace{\frac{Q_k}{Q}}_{\text{structure } s_k} \cdot x_{2k} \cdot x_{3k} \cdot x_{4k}$$

* **Activity effect** ($Q$): the whole business got bigger.
* **Structure (mix) effect** ($s_k$): customers moved between products. Moving toward products with high claims per customer raises claims, even if nothing else changes.
* The remaining factors are **intensity** effects.

No new code is needed: it is still a sum of products, only with 5 factors. Because $Q \cdot s_k = x_{1k}$, the cell values do not change, so **activity + structure equals the old x1 effect exactly**.

In [14]:
d = df_cells[factors].copy()
d["Q"] = d.groupby(level="year").x1.transform("sum")   # total customers, same for every product in a year
d["s"] = d.x1 / d.Q                                     # product share of customers (sums to 1 per year)
factors_ids = ["Q", "s", "x2", "x3", "x4"]
d[factors_ids]

Q      s     x2     x3      x4
year product                                       
2022 Term Life  140000 0.4286 1.2000 0.0120  500000
     Health     140000 0.5714 1.5000 0.3500    8000
2023 Term Life  149000 0.4228 1.2200 0.0125  510000
     Health     149000 0.5772 1.5200 0.3600    8300

In [15]:
add_s, mult_s = decompose(d.loc[2022], d.loc[2023], factors_ids, kind="product")
display(summary(add_s, mult_s))

# activity + structure == the x1 effect from section 6
assert np.isclose(add_s.loc["TOTAL", ["Q", "s"]].sum(), add_c.loc["TOTAL", "x1"])
print(f"activity + structure = {add_s.loc['TOTAL', ['Q', 's']].sum():,.2f}   x1 effect in case C = {add_c.loc['TOTAL', 'x1']:,.2f}")

,effect,share_%,index
Q,"51,275,716.2965",45.5485,1.0643
s,"-2,589,290.3555",-2.3001,0.9969
x2,"12,412,672.8460",11.0262,1.0152
x3,"29,008,817.8074",25.7687,1.0359
x4,"22,465,943.4057",19.9566,1.0277
TOTAL,"112,573,860.0000",100.0000,1.1466


activity + structure = 48,686,425.94   x1 effect in case C = 48,686,425.94


**Reading the structure effect.** Health's share of customers rose slightly (57.1% to 57.7%). A Health customer brings fewer claims than a Term Life customer (about 4,200 vs. 7,200 per customer in 2022), so the mix shift *reduced* total claims, which gives a negative `s` effect. Without this split, the shift would be hidden inside "customer growth".

This is usually the most valuable insight from a decomposition: growth in volume vs. a change in mix vs. a change in behaviour.

## 9. How additive and multiplicative relate

For every factor $i$, LMDI-I gives:

$$\frac{\ln D_{x_i}}{\Delta V_{x_i}} = \frac{1}{L(V^T, V^0)} \quad \text{(the same constant for every factor)}$$

So **each factor's share of the additive change equals its share of $\ln(\text{growth})$**. The two forms always tell the same story, with the same ranking and the same proportions. Pick whichever unit your audience prefers (amounts or %). `compare` asserts this property for every decomposition in this notebook.

In [16]:
shares = pd.DataFrame({
    "share of additive change": add_s.loc["TOTAL"] / add_s.loc["TOTAL"].sum(),
    "share of ln(growth)":      np.log(mult_s.loc["TOTAL"]) / np.log(mult_s.loc["TOTAL"]).sum(),
})
shares

,share of additive change,share of ln(growth)
Q,0.4555,0.4555
s,-0.0230,-0.0230
x2,0.1103,0.1103
x3,0.2577,0.2577
x4,0.1996,0.1996


## 10. Chained vs. one-step decomposition

Over several years you can either:

* **chain**: decompose each year-to-year step and sum the steps (multiply them for the multiplicative form), or
* **one-step**: decompose the first year against the last year directly.

Both add up exactly to the same total change, but the factor effects differ slightly, because the weights $L(\cdot)$ are computed on different pairs of periods. Chaining follows the actual path and is usually preferred for annual data. Whichever you choose, state it in your report.

In [17]:
years = list(df_prod.index)
one = lambda t0, t1: decompose(df_prod.loc[[t0]], df_prod.loc[[t1]].set_axis([t0]), factors)[0].loc["TOTAL"]

chained = sum(one(t0, t1) for t0, t1 in zip(years[:-1], years[1:]))
onestep = one(years[0], years[-1])

steps = pd.DataFrame({"chained": chained, "one-step": onestep, "difference": chained - onestep})
steps.loc["TOTAL"] = steps.sum()
steps

,chained,one-step,difference
x1,"30,595,118.8331","31,132,749.4158","-537,630.5827"
x2,"7,408,003.8729","7,098,837.9308","309,165.9421"
x3,"27,224,068.5235","26,288,785.2555","935,283.2680"
x4,"27,326,308.7705","28,033,127.3979","-706,818.6274"
TOTAL,"92,553,500.0000","92,553,500.0000",0.0000


## 11. Zero values and new products

`ln(0)` is undefined, so LMDI breaks when a value is zero. This happens when a product launches or closes, or when a claim type is 0 in one year. The standard fix (Ang & Liu, 2007) is to **replace zeros with a tiny $\delta$** (e.g. $10^{-10}$ to $10^{-20}$). The results converge as $\delta \to 0$.

Also, a product that exists in only one year has no row in the other year, so first **reindex both years to the union of cells**, then fill with $\delta$.

In [18]:
rows_new = [
    (2022, "Term Life", 60_000, 1.20, 0.0120, 500_000),
    (2022, "Health",    80_000, 1.50, 0.3500,   8_000),
    (2023, "Term Life", 63_000, 1.22, 0.0125, 510_000),
    (2023, "Health",    86_000, 1.52, 0.3600,   8_300),
    (2023, "Critical",   5_000, 1.00, 0.0500, 200_000),   # launched in 2023: no 2022 row
]
df_new = pd.DataFrame(rows_new, columns=["year", "product", *factors]).set_index(["year", "product"])


def align(df, t0, t1, factors, eps=1e-10):
    """Reindex both periods to the union of cells and replace zeros / missing with ``eps``."""
    cells = df.loc[t0].index.union(df.loc[t1].index)
    d0 = df.loc[t0, factors].reindex(cells).fillna(0).replace(0, eps)
    d1 = df.loc[t1, factors].reindex(cells).fillna(0).replace(0, eps)
    return d0, d1


d0, d1 = align(df_new, 2022, 2023, factors)
add_n, mult_n = decompose(d0, d1, factors)
add_n

,x1,x2,x3,x4
Critical,"14,359,817.4165","10,482,406.6422","9,118,614.0954","16,039,161.8459"
Health,"26,224,270.3901","4,802,865.4617","10,215,071.0262","13,349,153.1219"
Term Life,"22,462,155.5509","7,609,807.3842","18,793,746.7811","9,116,790.2838"
TOTAL,"63,046,243.3575","22,895,079.4882","38,127,431.9028","38,505,105.2516"


In [19]:
# Convergence: the new product's effect as delta shrinks
conv = {}
for eps in [1e-5, 1e-10, 1e-20, 1e-40]:
    d0, d1 = align(df_new, 2022, 2023, factors, eps)
    conv[f"{eps:.0e}"] = decompose(d0, d1, factors)[0].loc["Critical"]
conv = pd.DataFrame(conv).T.rename_axis("delta")
conv["cell total"] = conv.sum(axis=1)
conv

,x1,x2,x3,x4,cell total
delta,,,,,
1e-05,"15,702,695.8096","9,025,606.3659","6,677,089.4437","18,594,608.3808","50,000,000.0000"
1e-10,"14,359,817.4165","10,482,406.6422","9,118,614.0954","16,039,161.8459","50,000,000.0000"
1e-20,"13,511,544.0941","11,402,642.4706","10,660,883.4764","14,424,929.9589","50,000,000.0000"
1e-40,"13,028,991.7711","11,926,131.6867","11,538,225.3103","13,506,651.2318","50,000,000.0000"


**What this shows.** The new product's whole value (50M) is correctly included in the total. But as $\delta \to 0$ it is split **equally** across the 4 factors (12.5M each, approached slowly). The math cannot tell which factor "caused" a product to appear from nothing.

**Practical advice:** use $\delta$ so the code runs and the totals stay exact, but report new or closed products as a separate **entry / exit effect** (e.g. "+50M from the Critical Illness launch") rather than spreading them across the factors.

**Negative values** (e.g. profit, net cash flow) are not supported by plain LMDI, since logs need positive values. Split the item into positive parts (revenue − cost) or use a different method.

## 12. Cheat sheet

**Set up the identity first.** Write $V = \sum_k \prod_i x_{ik}$ and check that it holds exactly for your data (it should reproduce the reported total). Ratios like policies/customer or claims/policy are designed so that they cancel out.

| Situation | Cell value `a_tot` | Grand total `y_a` | Combine cells |
|---|---|---|---|
| one product of factors | `a.prod()` | omit | (one cell) |
| sum of products | `a.prod()` per cell | Σ cell values | add: sum, mult: product |
| sum of terms | `a` (each term) | Σ all terms | add: sum, mult: product |

**Pitfalls**

1. The cell value must equal the product of the factors you take logs of.
2. Several cells: always pass the grand total for the multiplicative form.
3. Per-cell multiplicative values are not interpretable; only `TOTAL` is.
4. Pure sums: additive LMDI = raw differences, so LMDI adds nothing there.
5. Zeros: reindex to the union of cells, fill with a small $\delta$, and report entry/exit separately.
6. Negative values are not supported.
7. Chained and one-step results differ: say which one you use.
8. Consider splitting a count into activity × structure to expose the mix effect.

## Exercises

1. **Mix only.** Change the 2023 Term Life and Health customers so that total customers stay at 140,000 but Term Life's share rises to 50%. Keep everything else as in 2022. Predict the sign of each effect, then check with `decompose`.
2. **Structure in case B.** Rewrite `df_sum` as total claims × the share of each line of business. What do you get, and why is it not very informative? (Hint: count the factors in each cell.)
3. **Chain the multiplicative form.** Extend section 10 so it also chains `mult` (multiply the steps). Check that the product of the chained indexes equals $y_{2025} / y_{2022}$.
4. **Region × product.** Build a dataset with 2 regions × 2 products. Make the cells `(region, product)` and decompose into activity, region share, product share within region, and claims per customer. Check that `compare` passes.
5. **Exit.** Remove Health in 2023 (product closed) and decompose with `align`. How is the lost value split across the factors?

## References

* Ang, B.W. (2004). *Decomposition analysis for policymaking in energy: which is the preferred method?* Energy Policy 32, 1131-1139.
* Ang, B.W. & Liu, N. (2007). *Handling zero values in the logarithmic mean Divisia index decomposition approach.* Energy Policy 35, 238-246.
* Ang, B.W. (2015). *LMDI decomposition approach: A guide for implementation.* Energy Policy 86, 233-238.